---
title: "Chapter 9: Hierarchical Parsing"
short_title: "9. Hierarchical parsing"
---

This chapter describes how each version of a paper is read into **one tree**: find the root file that LaTeX
compiles, follow the `\input`/`\include` commands to obtain the whole text in reading order, then split the text
into heading levels, paragraphs and leaf elements. The numbers at the end of the chapter are counted over every
version of the five ranges.

## 1. Finding the root file and merging multiple files (`gatherer.py`)

An arXiv version often consists of several `.tex` files. `gatherer.py` selects the **root file** the way TeX and
arXiv read the source:

- A candidate is a `.tex` file that is **not included by any other file** and has (or includes) a
  `\documentclass` (or LaTeX 2.09's `\documentstyle`) outside comments.
- When there are several candidates (figures built with the `standalone` class, subfiles, cover letters, old
  drafts), they are ranked by: a class other than `standalone`/`subfiles`, presence of `\begin{document}`, a usual
  name (`main`, `ms`, `paper`…), position in the top folder, the number of files it pulls in, and size.
- Without any `\documentclass`, the single `.tex` file, or the largest file with `\begin{document}`, is taken; such
  a guess is recorded in the version's list of problems.

From the root file, `\input{f}`, `\input f`, `\include{f}`, `\InputIfFileExists`, `\subfile`, `\import` and
`\subimport` are replaced by the content of the corresponding file, **in reading order**. File names are resolved
from the version's top folder first (arXiv always compiles from there), adding the `.tex` extension as TeX does.
Comments, `\iffalse` blocks and `comment` environments are removed at this step. The result of each version (root
file, files read, files referenced but missing, `.tex` files outside the compilation) is stored in
`parsed/gathering.json`.

## 2. Building the tree (`hierarchy.py`)

The merged text (after the cleaning and normalization of [Chapter 10](./10_standardization.ipynb)) is split into
nodes:

| Node kind | LaTeX source | Stored content |
| :--- | :--- | :--- |
| `document` | root of each version | `Version n root` |
| `heading` | `\part`, `\chapter`, `\section`, `\subsection`, `\subsubsection` (starred or not); `Abstract`, `Acknowledgements`, `Appendix` | the title |
| `paragraph` | `\paragraph`, `\subparagraph` | the title |
| `list`, `item` | `itemize`, `enumerate`, `description` and variants; `\item` | environment name; item label |
| `sentence`, `formula`, `figure` | the leaf elements of each paragraph | the normalized content |

**Level detection:** a node is always attached to the **deepest open unit**: a `\subsection` lies under the nearest
`\section`, a paragraph under the nearest heading or `\item`, a list under the unit that contains it. The sections
of an appendix lie under an `Appendix` node so that they can still be found. The preamble and the bibliography do
not enter the tree (references go to the `bibtex` table, [Chapter 11](./11_references.ipynb)). A version without TeX
still has its root node, with no children.

Each node receives an `element_id` from Member 4 (see [Chapter 12](./12_results.ipynb)), and each edge (child,
parent, version) is a row of the `hierarchy` table.

In [ ]:
import pandas as pd
import plotly.express as px
from report_utils_m2 import (COLORS, KIND_LABELS, PROBLEM_LABELS, group, human_bytes, human_seconds, load,
                             members, meta, metric, metric_table, num, ordered, pct, percent, ready, secs,
                             show, table)
READY = ready()

## 3. Status of the versions

In [ ]:
if READY:
    v = load("versions")
    names = {"ok": "TeX, tree built", "no_source": "No TeX in Milestone 1", "no_tex": "Folder without .tex"}
    v["Status"] = v["status"].map(lambda s: names.get(s, s))
    st = ordered(v.groupby(["member", "Status"]).size().reset_index(name="Versions"))
    fig = px.bar(st, x="member", y="Versions", color="Status", color_discrete_sequence=COLORS,
                 title="Status of the versions per range", labels={"member": "Range"})
    show(fig)
    total = v["Status"].value_counts().rename_axis("Status").reset_index(name="Versions")
    total["Share"] = (total["Versions"] / total["Versions"].sum()).map(pct)
    table(total, "Table 9.1. Status of the versions, whole group")

## 4. Versions made of several files

Number of `.tex` files on the compilation path of each version with TeX (the root file and every file it
includes, recursively):

In [ ]:
if READY:
    tex = v[v["status"] == "ok"].copy()
    bins = [0, 1, 2, 3, 5, 10, float("inf")]
    labels = ["1", "2", "3", "4–5", "6–10", "> 10"]
    tex[".tex files"] = pd.cut(tex["tex_files"], bins=bins, labels=labels)
    dist = tex.groupby(["member", ".tex files"], observed=False).size().reset_index(name="Versions")
    fig = px.bar(ordered(dist), x=".tex files", y="Versions", color="member", barmode="group",
                 color_discrete_sequence=COLORS, title="Number of .tex files on the compilation path of each version",
                 labels={"member": "Range"})
    show(fig)
    n = len(tex)
    rows = [
        ("Versions with TeX", f"{n:,}"),
        ("Made of 2 or more .tex files", f"{int(tex['multi_file'].sum()):,} ({pct(tex['multi_file'].mean())})"),
        ("Most files in one version", f"{int(tex['tex_files'].max()):,}"),
        ("With an included file missing from the source", f"{int((tex['missing_inputs'] > 0).sum()):,} ({pct((tex['missing_inputs'] > 0).mean())})"),
        ("With .tex files outside the compilation path", f"{int((tex['unused_tex'] > 0).sum()):,} ({pct((tex['unused_tex'] > 0).mean())})"),
    ]
    table(pd.DataFrame(rows, columns=["Metric", "Value"]), "Table 9.2. Merging multiple files, versions with TeX")

`.tex` files outside the compilation path are usually old drafts, cover letters or TikZ figures compiled
separately; they do not enter the tree, exactly as when arXiv compiles the paper.

## 5. Heading levels and tree depth

The **level** of a heading is the number of headings above it plus 1: a heading directly under the root is level 1
(usually a `\section`, or a `\chapter`/`\part` in theses and books). Levels are computed from the edges of the
`hierarchy` table, not from the command names, so they reflect the structure of the tree that was built.

In [ ]:
if READY:
    h = load("headings")
    h["Level"] = h["level"].map(lambda l: f"Level {l}")
    fig = px.bar(ordered(h), x="Level", y="headings", color="member", barmode="group", color_discrete_sequence=COLORS,
                 title="Headings per level", labels={"headings": "Headings", "member": "Range"})
    show(fig)
    lv = h.groupby("level")["headings"].sum().reset_index()
    lv["Share"] = (lv["headings"] / lv["headings"].sum()).map(pct)
    lv["headings"] = lv["headings"].map(num)
    table(lv.rename(columns={"level": "Level", "headings": "Headings"}), "Table 9.3. Headings per level, whole group")

In [ ]:
if READY:
    fig = px.histogram(ordered(tex), x="max_depth", color="member", barmode="group", color_discrete_sequence=COLORS,
                       title="Tree depth (edges from the root to the deepest node), versions with TeX",
                       labels={"max_depth": "Depth", "member": "Range"})
    fig.update_layout(yaxis_title="Versions")
    show(fig)
    d = tex["max_depth"]
    table(pd.DataFrame({"Statistic": ["Mean", "Median", "95th percentile", "Maximum"],
                        "Depth": [num(d.mean(), 2), num(d.median(), 1), num(d.quantile(0.95), 1), num(d.max())]}),
          "Table 9.4. Tree depth")

## 6. Problems recorded while merging files

`gatherer.py` records anything worth reviewing in a version (several root candidates, include cycles, missing
files, a guessed root file…). The table below counts each kind of problem. These problems do not break the tree:
the version is still built from the text that could be read, and `validate_m2.py` reports them as `WARN`
([Chapter 14](./14_validation.ipynb)).

In [ ]:
if READY:
    p = load("problems")
    if not p.empty:
        p["kind"] = p["kind"].map(lambda k: PROBLEM_LABELS.get(k, k))
        p = p[p["kind"] != "No TeX (Milestone 1)"]  # already counted in Table 9.1
    if p.empty:
        print("No problem was recorded.")
    else:
        pv = p.pivot_table(index="kind", columns="member", values="versions", aggfunc="sum", fill_value=0)
        pv = pv[[m for m in members()["member"] if m in pv.columns]]
        pv["Group"] = pv.sum(axis=1)
        pv.columns.name = None
        pv = pv.astype(int).sort_values("Group", ascending=False).reset_index().rename(columns={"kind": "Problem"})
        table(pv, "Table 9.5. Problems while merging files, per range (occurrences)")